# ADS Homework 3 - Part 2: CNN (PyTorch)


This notebook follows `.cursor/rules/task_description_hw3.mdc` and the plan in `hw3_plan.md`.


**Dataset (Kaggle path):**
- **Flowers-102**: `/kaggle/input/pytorch-challange-flower-dataset`

We use PyTorch for CNN experiments and transfer learning on image data.

**Author:** [Your Name]


In [ ]:
# Core imports and setup
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


## 1) Dataset Path
Confirm this path matches the Kaggle mount.

- **Flowers-102**: `/kaggle/input/pytorch-challange-flower-dataset`


In [ ]:
# Update if your Kaggle path differs
FLOWERS_ROOT = Path("/kaggle/input/pytorch-challange-flower-dataset")

print("Flowers-102 exists:", FLOWERS_ROOT.exists())


# Part 2: CNN on Flowers-102 (Image Classification)

**Task:** Multi-class classification (102 classes).

**Model A (Custom CNN):** Conv layers -> pooling -> fully connected.

**Experiments (comment on capacity, over/underfitting, training time, performance):**
- Kernel size (receptive field)
- Strides
- Number of filters
- Pooling type and pooling window size (max vs avg)
- Depth of the network

**Data Augmentation:** Random flips, rotations, crops, normalization and/or color jitter; analyze impact on overfitting/generalization.

**Model B (Transfer Learning):** Choose one pretrained model (e.g., ResNet18, VGG19). Clearly state frozen layers (if any) and compare performance to the custom CNN.


In [ ]:
import torchvision
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18, ResNet18_Weights

# 1. Setup Data with Augmentation
train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Attempt to find data path structure
def get_flower_loaders(root_path, batch_size=32):
    if not root_path.exists():
        print("Flowers root not found.")
        return None, None, 0
    
    # Check for train/valid folders
    train_dir = root_path / 'train'
    val_dir = root_path / 'valid'
    
    if not train_dir.exists():
        # Fallback: single folder, split it
        print("Train folder not explicitly found, checking subdirs...")
        # (Simple logic: if just one folder with classes, use Subset)
        # Assuming standard Kaggle structure for this dataset often has 'train'/'valid'/'test'
        # If not, allow user to adjust path manually.
        return None, None, 0

    train_ds = ImageFolder(train_dir, transform=train_tfms)
    val_ds = ImageFolder(val_dir, transform=val_tfms)
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    
    return train_loader, val_loader, len(train_ds.classes)

# If paths exist, create loaders
if FLOWERS_ROOT.exists():
    # Adjust if dataset is nested
    # (Logic from previous notebook kept short)
    train_loader_img, val_loader_img, num_classes = get_flower_loaders(FLOWERS_ROOT)
    if train_loader_img is None:
        print("Could not automatically load Flowers dataset structure. Please check paths.")
else:
    print("Flowers dataset not found.")


In [ ]:
class CustomCNN(nn.Module):
    def __init__(self, num_classes, base_channels=32, kernel_size=3, pool_type='max'):
        super().__init__()
        padding = kernel_size // 2
        
        def make_block(in_c, out_c):
            layers = [
                nn.Conv2d(in_c, out_c, kernel_size, padding=padding),
                nn.ReLU(),
                nn.BatchNorm2d(out_c)
            ]
            if pool_type == 'max':
                layers.append(nn.MaxPool2d(2))
            elif pool_type == 'avg':
                layers.append(nn.AvgPool2d(2))
            return layers

        self.features = nn.Sequential(
            *make_block(3, base_channels),
            *make_block(base_channels, base_channels*2),
            *make_block(base_channels*2, base_channels*4),
            *make_block(base_channels*4, base_channels*8),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Linear(base_channels*8, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.classifier(x)

def train_cnn(model, train_loader, val_loader, epochs=5, lr=1e-3):
    if train_loader is None: return
    model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    for epoch in range(1, epochs+1):
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        # Validation
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                preds = model(xb).argmax(dim=1)
                correct += (preds == yb).sum().item()
                total += yb.size(0)
        
        print(f"Epoch {epoch} | Train Loss {train_loss/len(train_loader):.4f} | Val Acc {correct/total:.4f}")


### CNN Experiments (based on the plan)
1. **Kernel size** (receptive field)
2. **Strides**
3. **Number of filters**
4. **Pooling type + window size** (max vs avg)
5. **Depth** (number of conv blocks)
6. **Transfer Learning** (feature extraction vs fine-tuning; note frozen layers)


In [ ]:
if FLOWERS_ROOT.exists() and train_loader_img:
    print("--- CNN Experiments ---")
    
    print("\n1. Custom CNN (Kernel=3)")
    cnn_base = CustomCNN(num_classes, kernel_size=3)
    train_cnn(cnn_base, train_loader_img, val_loader_img, epochs=5)
    
    print("\n2. Custom CNN (Kernel=5)")
    cnn_k5 = CustomCNN(num_classes, kernel_size=5)
    train_cnn(cnn_k5, train_loader_img, val_loader_img, epochs=5)
    
    print("\n3. Transfer Learning (ResNet18)")
    try:
        weights = ResNet18_Weights.IMAGENET1K_V1
        resnet = resnet18(weights=weights)
    except:
        resnet = resnet18(pretrained=True)
        
    # Replace head
    resnet.fc = nn.Linear(resnet.fc.in_features, num_classes)
    
    # Fine-tune all layers (or freeze by setting requires_grad=False on parameters)
    train_cnn(resnet, train_loader_img, val_loader_img, epochs=5, lr=1e-4)


### Discussion Question (CNN)
* **Why are CNNs fundamentally more parameter-efficient than MLPs for images?**
* **Under what conditions could an MLP match CNN performance, and why is this unrealistic in practice?**

*(Double-click to edit)*


## Wrap-up
- Summarize key findings.
- Optional: short error analysis.
